In [ ]:
"""
BOCD Trend Detection — Bayesian Online Change Point Detection on TSLA log returns.

Detects regime changes (trend shifts) in real-time using the Adams & MacKay algorithm
with a Gaussian observation model and normal-inverse-gamma conjugate prior.
"""

import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.append(os.path.abspath(os.path.join('/Users/nikolastsalidis/Desktop/smart-money-concepts', 'src')))
from data_loader import DataLoader

# --- Load data ---
symbol = 'TSLA'
timeframe = '15min'

loader = DataLoader(symbol)
df = loader.get_data(timeframe)
close = df['close'].values
log_returns = np.log(close[1:] / close[:-1])
times = df['time'].values[1:]

print(f"Loaded {len(log_returns)} log returns for {symbol} {timeframe}")

In [ ]:
def bocd(data, hazard_rate=1/100, mu0=0.0, var0=1.0, obs_var=None):
    """
    Bayesian Online Change Point Detection (Adams & MacKay 2007).

    Known-variance Gaussian model (unknown mean only).

    Returns the MAP run length at each timestep. A changepoint is detected
    when the MAP run length drops significantly (resets to a small value).

    Args:
        data: 1D array of observations (log returns)
        hazard_rate: 1/expected_run_length
        mu0: prior mean
        var0: prior variance on the mean
        obs_var: observation variance (if None, estimated from data)

    Returns:
        map_rl: MAP run length at each timestep
        growth_probs: growth probability (sum of R for run_length > 0)
    """
    T = len(data)
    if obs_var is None:
        obs_var = np.var(data)

    map_rl = np.zeros(T, dtype=int)
    R = np.array([1.0])

    post_mean = np.array([mu0])
    post_prec = np.array([1.0 / var0])

    H = hazard_rate
    obs_prec = 1.0 / obs_var
    MAX_RL = 500

    for t in range(T):
        x = data[t]

        pred_var = obs_var + 1.0 / post_prec
        log_pred = -0.5 * np.log(2 * np.pi * pred_var) - 0.5 * (x - post_mean)**2 / pred_var
        pred = np.exp(log_pred)

        growth = R * pred * (1 - H)
        cp = np.sum(R * pred * H)

        new_R = np.empty(len(growth) + 1)
        new_R[0] = cp
        new_R[1:] = growth

        evidence = new_R.sum()
        if evidence > 0:
            new_R /= evidence

        map_rl[t] = np.argmax(new_R)

        if len(new_R) > MAX_RL:
            new_R = new_R[:MAX_RL]
            new_R /= new_R.sum()

        R = new_R

        new_prec = post_prec + obs_prec
        new_mean = (post_prec * post_mean + obs_prec * x) / new_prec

        post_mean = np.concatenate([[mu0], new_mean])[:MAX_RL]
        post_prec = np.concatenate([[1.0 / var0], new_prec])[:MAX_RL]

    return map_rl


def detect_changepoints(map_rl, min_drop=10):
    """
    Detect changepoints from MAP run-length sequence.

    A changepoint occurs when MAP run length drops by at least `min_drop`.
    """
    cp_indices = []
    for t in range(1, len(map_rl)):
        if map_rl[t] < map_rl[t-1] - min_drop:
            cp_indices.append(t)
    return np.array(cp_indices)

print("BOCD functions defined.")

In [ ]:
# --- Run BOCD and extract changepoints + trends ---

ret_std = np.std(log_returns)
print(f"Log return std: {ret_std:.6f}")

hazard = 1 / 100
map_rl = bocd(log_returns, hazard_rate=hazard,
              mu0=0.0, var0=ret_std**2 * 10, obs_var=ret_std**2)

print(f"MAP run length stats: min={map_rl.min()}, max={map_rl.max()}, mean={map_rl.mean():.1f}")

# Detect changepoints: where MAP run length drops significantly
cp_indices = detect_changepoints(map_rl, min_drop=10)
print(f"Detected {len(cp_indices)} changepoints")

# Label trend direction per segment
segments = np.split(np.arange(len(log_returns)), cp_indices)
trend = np.zeros(len(log_returns), dtype=int)
for seg in segments:
    if len(seg) == 0:
        continue
    mean_ret = log_returns[seg].mean()
    trend[seg] = 1 if mean_ret > 0 else -1

result = pd.DataFrame({
    'time': times,
    'close': close[1:],
    'log_return': log_returns,
    'map_run_length': map_rl,
    'is_changepoint': np.isin(np.arange(len(log_returns)), cp_indices),
    'trend': trend,
})

result.head(10)

In [ ]:
# --- Visualization: price colored by trend + MAP run length ---

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# Top: close price colored by trend
for i in range(len(result) - 1):
    color = '#2ecc71' if result['trend'].iloc[i] == 1 else '#e74c3c'
    ax1.plot(result['time'].iloc[i:i+2], result['close'].iloc[i:i+2],
             color=color, linewidth=0.8)

# Vertical lines at changepoints
for idx in cp_indices:
    ax1.axvline(result['time'].iloc[idx], color='blue', alpha=0.4, linewidth=0.7, linestyle='--')

ax1.set_ylabel('Close Price')
ax1.set_title(f'{symbol} {timeframe} — BOCD Trend Detection (hazard=1/{int(1/hazard)}, {len(cp_indices)} changepoints)')
ax1.grid(True, alpha=0.3)

# Bottom: MAP run length over time
ax2.fill_between(result['time'], result['map_run_length'], alpha=0.4, color='steelblue')
ax2.set_ylabel('MAP Run Length')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- Sensitivity analysis: vary hazard rate ---

hazard_values = [1/50, 1/100, 1/250]
fig, axes = plt.subplots(len(hazard_values), 1, figsize=(16, 4 * len(hazard_values)), sharex=True)

for ax, h in zip(axes, hazard_values):
    rl_h = bocd(log_returns, hazard_rate=h,
                mu0=0.0, var0=ret_std**2 * 10, obs_var=ret_std**2)
    cp_h = detect_changepoints(rl_h, min_drop=10)
    n_cp = len(cp_h)

    ax.plot(times, close[1:], color='gray', linewidth=0.5, alpha=0.6)
    ax_rl = ax.twinx()
    ax_rl.fill_between(times, rl_h, alpha=0.2, color='steelblue')
    ax_rl.set_ylabel('MAP RL')

    for idx in cp_h:
        ax.axvline(times[idx], color='blue', alpha=0.3, linewidth=0.5, linestyle='--')

    ax.set_title(f'hazard=1/{int(1/h)} — {n_cp} changepoints')
    ax.set_ylabel('Close')
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{symbol} {timeframe} — BOCD Sensitivity Analysis', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()